# Safety-First: When the Agent Overrides

**Demonstrating 5 safety guardrails that override normal clinical pathways in the CAP CDSS pipeline.**

> This notebook is generated by `_generate_safety_overrides_notebook.py`. Do not edit manually.

### Why Safety Overrides Matter

Clinical safety is the #1 concern for medical AI. A decision support system that only follows
the "happy path" is dangerous — real patients present with edge cases, missing data, and
contradictory signals that demand the system err on the side of caution.

This notebook demonstrates 5 scenarios where the CAP CDSS agent **overrides expected pathways**
to protect patient safety. The system's core safety principle:

> **Overrides can only ESCALATE, never suppress.**

Alert tiering ensures clinicians see the right information at the right urgency:
- **High/moderate confidence** alerts require clinician action
- **Low confidence** alerts are informational — for awareness, never suppressed

### Scenarios Demonstrated

| # | Scenario | Override | Key Insight |
|---|----------|----------|-------------|
| 1 | Sepsis recognition | Low CURB65 → hospital | Low severity score does NOT mean safe to discharge |
| 2 | Allergy classification | Intolerance vs true allergy | "Penicillin allergy" is NOT binary |
| 3 | Discharge block | Improving → still unsafe | Getting better does NOT mean ready to go home |
| 4 | Missing data | CURB65 → CRB65 fallback | Report uncertainty rather than guess |
| 5 | Oral intolerance | Oral → IV route | Functional status matters, not just severity scores |

### Quick Start
1. Set runtime to **GPU -> A100** (Runtime -> Change runtime type)
2. Add **HF_TOKEN** to Colab Secrets (key icon in left sidebar)
3. *(Optional)* Add **GITHUB_TOKEN** for private repo install
4. **Run All** (Runtime -> Run all)


> **Disclaimer:** This notebook demonstrates a research prototype. All clinical outputs
> (severity scores, antibiotic recommendations, contradiction alerts, CXR analysis)
> are **AI-generated by MedGemma 1.5 4B** and have been validated only on synthetic data.
> This system is **not a medical device**, is not approved for clinical use, and must not
> be used for patient care decisions. See [DISCLAIMER.md](../DISCLAIMER.md) for full terms.


In [ ]:
# Cell 1: Install package + dependencies
import os

# Detect Colab vs local
try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Install from GitHub (requires GITHUB_TOKEN in Colab Secrets)
    try:
        github_token = userdata.get("GITHUB_TOKEN")
        repo_url = f"git+https://{github_token}@github.com/HP-00/MedGemma-Pneumonia-Agent.git@main"
    except Exception:
        repo_url = "git+https://github.com/HP-00/MedGemma-Pneumonia-Agent.git@main"

    %pip install --quiet {repo_url}
    %pip install --quiet plotly>=5.0.0 pandas>=2.0.0 nest-asyncio
else:
    print("Local environment detected. Ensure: pip install -e '.[dev,benchmark]'")

import nest_asyncio
nest_asyncio.apply()

print("Setup complete")


## Authentication


In [ ]:
import os

# HuggingFace token for gated model access
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except ImportError:
    pass

assert os.environ.get("HF_TOKEN"), "HF_TOKEN not set! Add it to Colab Secrets or environment."
print("Authentication complete")


## Load Model & Build Graph


In [ ]:
import time
from cap_agent.models.medgemma import get_model_and_processor
from cap_agent.agent.graph import build_cap_agent_graph
from cap_agent.agent.state import build_initial_state

# Load MedGemma (lazy singleton - only loads once, includes warm-up)
model, processor = get_model_and_processor()
print(f"Model loaded on {model.device}")

# Build the 8-node LangGraph pipeline
graph = build_cap_agent_graph()
print("Graph compiled: 8 nodes, conditional routing at contradiction check")


## Safety-First Design Philosophy

The CAP CDSS pipeline implements **one-directional overrides**: safety mechanisms can only
**escalate** care, never suppress alerts or downgrade recommendations. A false positive
(unnecessary caution) is always preferred over a missed safety issue.

### Alert Tiering

| Confidence | Action | Example |
|------------|--------|---------|
| **High** | Clinician action required | Sepsis markers with low CURB65 |
| **Moderate** | Clinician review recommended | Fluoroquinolone for penicillin intolerance |
| **Low** | Informational — for awareness | Borderline lab values |

### 5 Safety Mechanisms Demonstrated

1. **Sepsis recognition** overrides low severity scores when sepsis markers detected
2. **Allergy classification** distinguishes true allergy from intolerance to prevent inappropriate prescribing
3. **Treatment reassessment** blocks premature discharge despite improving vitals
4. **Missing data handling** triggers explicit uncertainty reporting and CRB65 fallback
5. **Functional status** (oral tolerance) affects treatment route and place of care


In [ ]:
# Clinical output renderer — formats structured_output as a readable document
def render_clinical_output(result, scenario_title="Pipeline Output"):
    """Render structured_output as a formatted clinical document using ANSI codes."""
    ESC = chr(27)
    B = f"{ESC}[1m"       # Bold
    R = f"{ESC}[91m"      # Red
    Y = f"{ESC}[93m"      # Yellow
    G = f"{ESC}[92m"      # Green
    C = f"{ESC}[96m"      # Cyan
    X = f"{ESC}[0m"       # Reset

    so = result.get("structured_output", {})
    sections = so.get("sections", {})

    print(f"\n{B}{C}{'=' * 70}")
    print(f"  CLINICAL DECISION SUPPORT — {scenario_title}")
    print(f"{'=' * 70}{X}")
    print(f"{Y}AI-generated observations for clinician review — not a substitute for clinical judgement.{X}\n")

    # 1. PATIENT
    s1 = sections.get("1_patient", {})
    demo = s1.get("demographics", {})
    print(f"{B}{C}1. PATIENT{X}")
    print(f"  {demo.get('age', '?')}yo {demo.get('sex', '?')}")
    allergy_list = demo.get("allergies", [])
    if allergy_list:
        for a in allergy_list:
            if isinstance(a, dict):
                print(f"  {R}Allergy: {a.get('drug', '?')} — {a.get('reaction_type', '?')} ({a.get('severity', '?')}){X}")
            else:
                print(f"  {R}Allergy: {a}{X}")
    else:
        print(f"  {G}No known drug allergies{X}")
    combos = demo.get("comorbidities", [])
    if combos:
        print(f"  Comorbidities: {', '.join(combos)}")
    if demo.get("pregnancy"):
        print(f"  {R}PREGNANT{X}")
    print(f"  Oral tolerance: {'Yes' if demo.get('oral_tolerance', True) else R + 'No' + X}")
    travel = demo.get("travel_history", [])
    if travel:
        print(f"  {Y}Travel: {', '.join(travel)}{X}")
    print()

    # 2. SEVERITY
    s2 = sections.get("2_severity", {})
    curb = s2.get("curb65", {})
    sev = curb.get("severity_tier", "?")
    sev_color = R if sev == "high" else (Y if sev == "moderate" else G)
    score = curb.get("curb65")
    score_str = f"CURB65={score}" if score is not None else f"CRB65={curb.get('crb65', '?')} (urea unavailable)"
    print(f"{B}{C}2. SEVERITY{X}")
    print(f"  {score_str}  {sev_color}{B}{sev.upper()}{X}")
    print(f"  C={curb.get('c','?')} U={curb.get('u','?')} R={curb.get('r','?')} B={curb.get('b','?')} 65={curb.get('age_65','?')}")
    missing = curb.get("missing_variables", [])
    if missing:
        print(f"  {Y}Missing: {', '.join(missing)}{X}")
    poc = s2.get("place_of_care", {})
    if poc:
        print(f"  Place of care: {poc.get('recommendation', '?')}")
    print()

    # 3. CXR
    s3 = sections.get("3_cxr", {})
    cxr = s3.get("findings", {})
    print(f"{B}{C}3. CHEST X-RAY{X}")
    for finding in ["consolidation", "pleural_effusion", "cardiomegaly", "edema", "atelectasis"]:
        f = cxr.get(finding, {})
        if not isinstance(f, dict):
            continue
        present = f.get("present", False)
        conf = f.get("confidence", "?")
        if present:
            loc = f.get("location", "")
            bbox = f.get("bounding_box")
            line = f"  {R}PRESENT{X} {finding} ({conf} confidence)"
            if loc:
                line += f" at {loc}"
            if bbox:
                line += f" [bbox: {bbox}]"
            print(line)
        else:
            print(f"  {G}absent{X}  {finding} ({conf} confidence)")
    iq = cxr.get("image_quality", {})
    if iq:
        print(f"  Quality: {iq.get('projection','?')} / {iq.get('rotation','?')} / {iq.get('penetration','?')}")
    longit = cxr.get("longitudinal_comparison")
    if longit:
        print(f"  {B}Longitudinal:{X}")
        for fn, ch in longit.items():
            if isinstance(ch, dict):
                print(f"    {fn}: {ch.get('change','?')} — {ch.get('description','')}")
    print()

    # 4. KEY BLOODS
    s4 = sections.get("4_key_bloods", {})
    labs = s4.get("values", {})
    print(f"{B}{C}4. KEY BLOODS{X}")
    for test, data in (labs or {}).items():
        if not isinstance(data, dict):
            continue
        val = data.get("value", "?")
        unit = data.get("unit", "")
        ref = data.get("reference_range", "")
        abn = data.get("abnormal_flag", False)
        color = R if abn else G
        flag = " ABNORMAL" if abn else ""
        print(f"  {color}{test}: {val} {unit}{flag}{X}  (ref: {ref})")
    print()

    # 5. CONTRADICTIONS
    s5 = sections.get("5_contradiction_alert", {})
    alerts = s5.get("alerts", [])
    informational = s5.get("informational", [])
    resolutions = s5.get("resolutions", [])
    n_total = s5.get("detected", 0)
    print(f"{B}{C}5. CONTRADICTION ALERTS{X}")
    if n_total == 0:
        print(f"  {G}No contradictions detected — findings concordant{X}")
    else:
        print(f"  {n_total} contradiction(s) detected")
        for c in alerts:
            conf = c.get("confidence", "?")
            badge_color = R if conf == "high" else Y
            print(f"  {badge_color}[{conf.upper()}]{X} {c.get('rule_id','?')}: {c.get('pattern','')}")
            if c.get("evidence_for"):
                print(f"    For: {c['evidence_for']}")
            if c.get("evidence_against"):
                print(f"    Against: {c['evidence_against']}")
            rec = c.get("recommendation")
            if rec:
                print(f"    {B}Recommendation:{X} {rec}")
        for c in informational:
            print(f"  {G}[LOW]{X} {c.get('rule_id','?')}: {c.get('pattern','')}")
        if resolutions:
            print(f"  {B}Resolutions:{X}")
            for r in resolutions:
                print(f"    {r[:200]}")
    print()

    # 6. TREATMENT
    s6 = sections.get("6_treatment_pathway", {})
    abx = s6.get("antibiotic", {})
    print(f"{B}{C}6. TREATMENT{X}")
    print(f"  First-line: {abx.get('first_line', 'N/A')}")
    print(f"  Dose/route: {abx.get('dose_route', 'N/A')}")
    if abx.get("allergy_adjustment"):
        print(f"  {Y}Allergy adj: {abx['allergy_adjustment']}{X}")
    if abx.get("atypical_cover"):
        print(f"  Atypical: {abx['atypical_cover']}")
    if abx.get("renal_adjustment"):
        print(f"  {Y}Renal: {abx['renal_adjustment']}{X}")
    cortico = s6.get("corticosteroid")
    if cortico:
        print(f"  Corticosteroid: {cortico}")
    steward = abx.get("stewardship_notes", [])
    for note in steward:
        print(f"  {Y}Stewardship: {note}{X}")
    inv = s6.get("investigations", {})
    if inv:
        print(f"  {B}Investigations:{X}")
        for name, detail in inv.items():
            if isinstance(detail, dict):
                rec = "Recommended" if detail.get("recommended") else "Not indicated"
                print(f"    {name}: {rec} — {detail.get('reasoning', '')[:100]}")
    print()

    # 7. DATA GAPS
    s7 = sections.get("7_data_gaps", {})
    gaps = s7.get("gaps", [])
    print(f"{B}{C}7. DATA GAPS{X}")
    if gaps:
        for g in gaps:
            print(f"  {Y}- {g}{X}")
    else:
        print(f"  {G}None identified{X}")
    print()

    # 8. MONITORING
    s8 = sections.get("8_monitoring", {})
    plan = s8.get("plan", {})
    print(f"{B}{C}8. MONITORING{X}")
    crp_trend = plan.get("crp_trend")
    if crp_trend:
        adm = crp_trend.get("admission_value", "?")
        cur = crp_trend.get("current_value", "?")
        pct = crp_trend.get("percent_change", "?")
        trend = crp_trend.get("trend", "?")
        sr = crp_trend.get("flag_senior_review", False)
        sr_color = R if sr else G
        print(f"  CRP: {adm} -> {cur} mg/L ({pct}% change, {trend})")
        print(f"  Senior review: {sr_color}{sr}{X}")
    tr = plan.get("treatment_response")
    if tr:
        reassess = tr.get("reassess_needed", False)
        ra_color = R if reassess else G
        print(f"  Treatment response: {ra_color}reassess_needed={reassess}{X}")
        for action in tr.get("actions", []):
            print(f"    - {action}")
    dc = plan.get("discharge_criteria_met")
    if dc is not None:
        dc_color = G if dc else R
        dc_str = "MET" if dc else "NOT MET"
        print(f"  Discharge: {dc_color}{dc_str}{X}")
    dc_detail = plan.get("discharge_criteria_details", {})
    if dc_detail:
        for check, passed_val in dc_detail.items():
            if check.startswith("_"):
                continue
            chk_color = G if passed_val else R
            chk_str = "PASS" if passed_val else "FAIL"
            print(f"    {chk_color}[{chk_str}]{X} {check}")
    print(f"  Next review: {plan.get('next_review', 'N/A')}")
    print()

    # PROVENANCE
    prov = so.get("provenance", {})
    if prov:
        print(f"{B}{C}PROVENANCE{X}")
        tools = prov.get("extraction_tools", {})
        sources = prov.get("data_sources", {})
        for tool_name, pipeline in tools.items():
            src = sources.get(tool_name, [])
            print(f"  {tool_name}: {pipeline} ({', '.join(src) if src else 'N/A'})")

    print(f"\n{C}{'=' * 70}{X}\n")

print("render_clinical_output() defined")


## Load Safety Cases

Three data sources provide the override scenarios:
- **Group D** (3 cases): Sepsis override, missing urea, oral intolerance
- **CR-10** (2 cases): Allergy classification — intolerance vs true allergy
- **Day 3-4** (1 case): Treatment reassessment blocks premature discharge


In [ ]:
from benchmark_data.cases.group_d_safety import get_group_d_cases
from benchmark_data.cases.registry import get_track2_cases
from cap_agent.data.synthetic import get_synthetic_day34_case

# Group D: safety edge cases
group_d = get_group_d_cases()
print(f"Group D safety cases: {len(group_d)}")
for c in group_d:
    print(f"  {c['case_id']}: CURB65={c['ground_truth']['curb65']}, "
          f"severity={c['ground_truth']['severity_tier']}")

# CR-10 allergy cases
all_track2 = get_track2_cases()
cr10_cases = [c for c in all_track2 if c["case_id"].startswith("CR10-")]
print(f"\nCR-10 allergy cases: {len(cr10_cases)}")
for c in cr10_cases:
    print(f"  {c['case_id']}: expected contradictions: {c['ground_truth'].get('contradictions', [])}")

# Day 3-4 discharge case
d34_case = get_synthetic_day34_case()
print(f"\nDay 3-4 case: {d34_case['case_id']}")


In [ ]:
all_results = []

all_cases = group_d + cr10_cases + [d34_case]
for i, case in enumerate(all_cases):
    case_id = case["case_id"]
    print(f"[{i+1}/{len(all_cases)}] Running {case_id}...")

    initial_state = build_initial_state(case)
    start = time.time()
    result = None
    for chunk in graph.stream(initial_state, stream_mode="values"):
        result = chunk
    elapsed = time.time() - start

    all_results.append({
        "case_id": case_id,
        "result": result,
        "elapsed": elapsed,
        "case": case,
    })
    n_contradictions = len(result.get("contradictions_detected", []))
    print(f"  Done in {elapsed:.1f}s — {n_contradictions} contradictions detected")

print(f"\nAll {len(all_results)} cases complete")


## Scenario 1: Sepsis Recognition Override

**Patient:** 45-year-old, CURB65=1 (low severity) — normally "consider discharge home".

**BUT:** Four sepsis markers trigger hospital override:
- Lactate = 3.5 mmol/L (>2)
- Heart rate = 110 bpm (>100)
- SBP = 85 mmHg (<90)
- Temperature = 38.5C (>38.3)

The `recommend_place_of_care()` function detects these markers and overrides the low severity
recommendation to **hospital referral**, regardless of the CURB65 score.

**Key insight:** A low CURB65 does NOT mean it is safe to discharge. Sepsis can kill
patients with otherwise "low severity" pneumonia.


In [ ]:
sepsis_r = [r for r in all_results if r["case_id"] == "SEPSIS-OVERRIDE"][0]
result = sepsis_r["result"]

# Show CURB65 score
curb = result.get("curb65_score", {})
print(f"CURB65: {curb.get('curb65')} — Severity: {curb.get('severity_tier', '?').upper()}")
print(f"  Normal recommendation would be: 'Consider discharge home'\n")

# Show sepsis markers
so = result.get("structured_output", {})
sections = so.get("sections", {})
s2 = sections.get("2_severity", {})
poc = s2.get("place_of_care", {})

print(f"Place of care: {poc.get('recommendation', '?')}")
if poc.get("sepsis_override"):
    print(f"  SEPSIS OVERRIDE ACTIVATED")
    markers = poc.get("sepsis_markers", [])
    for m in markers:
        print(f"    → {m}")
print(f"\nOptions: {poc.get('options', [])}")


In [ ]:
render_clinical_output(sepsis_r["result"], "Sepsis Recognition Override")


## Scenario 2: Allergy Safety — True Allergy vs Intolerance

Two patients with identical clinical presentations but different allergy histories:

**Patient A (CR10-FIRE):** "Penicillin allergy" with **GI upset** — this is an **intolerance**,
not a true allergy. The system detects that levofloxacin (a fluoroquinolone) was prescribed
unnecessarily. CR-10 fires, recommending beta-lactam reassessment.

**Patient B (CR10-TRUE):** "Penicillin allergy" with **anaphylaxis** — this is a **true
IgE-mediated allergy**. Levofloxacin is genuinely required. CR-10 does NOT fire.
An MHRA fluoroquinolone safety warning is still issued (standard practice).

**Key insight:** "Penicillin allergy" is NOT binary. Up to 90% of reported penicillin
"allergies" are intolerances. The classification determines whether a patient receives
a beta-lactam (safer, narrower spectrum) or a fluoroquinolone (broader spectrum, more
adverse effects including tendon rupture and aortic dissection).


In [ ]:
for r in [r for r in all_results if r["case_id"].startswith("CR10-")]:
    case_id = r["case_id"]
    result = r["result"]
    case = r["case"]

    print(f"\n{'=' * 50}")
    print(f"  {case_id}")
    print(f"{'=' * 50}")

    # Show allergy details
    allergies = case.get("past_medical_history", {}).get("allergies", [])
    for a in allergies:
        if isinstance(a, dict):
            print(f"  Allergy: {a.get('drug', '?')}")
            print(f"  Reaction: {a.get('reaction_type', '?')}")
            print(f"  Severity: {a.get('severity', '?')}")

    # Show antibiotic recommendation
    abx = result.get("antibiotic_recommendation", {})
    print(f"\n  Antibiotic: {abx.get('first_line', '?')}")
    print(f"  Route: {abx.get('route', '?')}")

    # Show contradictions (CR-10 should fire for FIRE, not for TRUE)
    contradictions = result.get("contradictions_detected", [])
    cr10 = [c for c in contradictions if c.get("rule_id") == "CR-10"]
    if cr10:
        print(f"\n  CR-10 FIRED: {cr10[0].get('description', '?')}")
        print(f"  Confidence: {cr10[0].get('confidence', '?')}")
    else:
        print(f"\n  CR-10 did NOT fire — fluoroquinolone is appropriate for true allergy")


In [ ]:
for r in [r for r in all_results if r["case_id"].startswith("CR10-")]:
    render_clinical_output(r["result"], f"Allergy Safety: {r['case_id']}")


## Scenario 3: Discharge Block Despite Improvement

**Patient:** Day 3-4 of treatment, showing clinical improvement:
- CRP trending down (186 → 110 mg/L)
- Temperature normalising (37.8C)

**BUT:** Treatment reassessment identifies remaining instability markers.
`reassess_needed=True` overrides discharge criteria — the patient stays in hospital
regardless of improving vitals.

This is a **one-directional override**: treatment reassessment can only BLOCK discharge,
never accelerate it.

**Key insight:** Getting better does NOT mean ready to go home. The system protects
against premature discharge by requiring all stability criteria to be met.


In [ ]:
d34_r = [r for r in all_results if r["case_id"] == d34_case["case_id"]][0]
result = d34_r["result"]

so = result.get("structured_output", {})
sections = so.get("sections", {})
s7 = sections.get("7_monitoring", {}) if "7_monitoring" in sections else sections.get("8_monitoring", {})

# Treatment response
treatment = s7.get("treatment_response", {})
print(f"Treatment response: {treatment.get('assessment', '?')}")
print(f"Reassessment needed: {treatment.get('reassess_needed', '?')}")
if treatment.get("reasons"):
    for reason in treatment["reasons"]:
        print(f"  → {reason}")

# Discharge criteria
discharge = s7.get("discharge_criteria", {})
print(f"\nDischarge criteria met: {discharge.get('all_met', 'N/A')}")
criteria = discharge.get("criteria", {})
for k, v in criteria.items():
    status = "PASS" if v else "FAIL"
    marker = "" if v else "  ← BLOCKING"
    print(f"  {k}: {status}{marker}")

if treatment.get("reassess_needed") and not discharge.get("all_met", True):
    print(f"\n  → DISCHARGE BLOCKED: Treatment reassessment overrides — safety first")


In [ ]:
render_clinical_output(d34_r["result"], "Discharge Block Despite Improvement")


## Scenario 4: Missing Data — Graceful Degradation

**Patient:** 72-year-old with no urea measurement available.

The system **cannot** compute full CURB65 (needs urea for the U component). Instead of
guessing or silently ignoring the gap, it:
1. Falls back to **CRB65** (4-point scale without urea)
2. Explicitly flags the **data gap** — tells clinicians what it does not know
3. Reports `curb65=None` to make the limitation visible in structured output

**Key insight:** The system reports uncertainty rather than guessing. Missing data
is surfaced, not hidden.


In [ ]:
missing_r = [r for r in all_results if r["case_id"] == "MISSING-UREA"][0]
result = missing_r["result"]

curb = result.get("curb65_score", {})
print(f"CURB65: {curb.get('curb65')} (None = cannot compute — urea missing)")
print(f"CRB65 (fallback): {curb.get('crb65')}")
print(f"Severity tier: {curb.get('severity_tier', '?').upper()} (based on CRB65)")

# Data gaps
missing_vars = curb.get("missing_variables", [])
if missing_vars:
    print(f"\nMissing variables flagged:")
    for m in missing_vars:
        print(f"  → {m}")

# Check structured output for data gap reporting
so = result.get("structured_output", {})
sections = so.get("sections", {})
s2 = sections.get("2_severity", {})
curb_section = s2.get("curb65", {})
missing_in_output = curb_section.get("missing_variables", [])
if missing_in_output:
    print(f"\nData gaps in clinical output:")
    for m in missing_in_output:
        print(f"  → {m}")

print(f"\n  → System correctly degrades: CURB65 unavailable → CRB65 fallback")
print(f"  → Missing data explicitly reported — no silent assumptions")


In [ ]:
render_clinical_output(missing_r["result"], "Missing Data — Graceful Degradation")


## Scenario 5: Oral Intolerance Override

**Patient:** 55-year-old, CURB65=0 (low severity) — normally outpatient with oral antibiotics.

**BUT:** The patient cannot swallow (vomiting/dysphagia). Oral antibiotics are not feasible,
so the system switches to IV route, which requires hospital admission consideration.

The `select_antibiotic()` function detects `oral_tolerance=False` and changes the treatment
route. `recommend_place_of_care()` adds hospital admission for IV therapy as an option.

**Key insight:** Functional status matters as much as clinical severity scores. A patient
with CURB65=0 who cannot swallow still needs hospital-level care.


In [ ]:
oral_r = [r for r in all_results if r["case_id"] == "ORAL-INTOLERANCE"][0]
result = oral_r["result"]

curb = result.get("curb65_score", {})
print(f"CURB65: {curb.get('curb65')} — Severity: {curb.get('severity_tier', '?').upper()}")
print(f"  Normal recommendation: outpatient with oral antibiotics\n")

# Show antibiotic route changed to IV
abx = result.get("antibiotic_recommendation", {})
print(f"Antibiotic: {abx.get('first_line', '?')}")
print(f"Route: {abx.get('route', '?')}")
if "IV" in str(abx.get("route", "")) or "IV" in str(abx.get("first_line", "")):
    print(f"  → Route changed to IV due to oral intolerance")

# Show place of care
so = result.get("structured_output", {})
sections = so.get("sections", {})
s2 = sections.get("2_severity", {})
poc = s2.get("place_of_care", {})
print(f"\nPlace of care: {poc.get('recommendation', '?')}")
options = poc.get("options", [])
for opt in options:
    if "IV" in opt or "oral intolerance" in opt.lower():
        print(f"  → {opt}")


In [ ]:
render_clinical_output(oral_r["result"], "Oral Intolerance Override")


## Summary: Override Taxonomy

A visual summary of the 5 override types demonstrated, showing the number of cases
per category and the nature of each override.


In [ ]:
import plotly.graph_objects as go

override_types = [
    "Sepsis\nRecognition",
    "Allergy\nClassification",
    "Discharge\nBlock",
    "Missing\nData",
    "Oral\nIntolerance",
]
descriptions = [
    "Low CURB65 → hospital",
    "Intolerance vs allergy",
    "Improving → still unsafe",
    "CURB65 → CRB65 fallback",
    "Oral → IV route",
]

fig = go.Figure(data=[
    go.Bar(
        x=override_types,
        y=[1, 2, 1, 1, 1],  # Number of cases per type
        text=descriptions,
        textposition="auto",
        marker_color=["#dc2626", "#f59e0b", "#dc2626", "#6366f1", "#f59e0b"],
    )
])

fig.update_layout(
    title="Safety Override Cases by Type",
    yaxis_title="Number of Cases",
    template="plotly_white",
    height=400,
    showlegend=False,
)
fig.show()


## Safety Score Verification

Every case must have a safety score of 1.0 — the system must never prescribe a drug
that the patient has a true allergy (anaphylaxis) to.


In [ ]:
print("=== SAFETY SCORE VERIFICATION ===\n")
print("Every case must have safety_score = 1.0 (no unsafe recommendations)\n")

for r in all_results:
    case_id = r["case_id"]
    so = r["result"].get("structured_output", {})
    sections = so.get("sections", {})
    # Safety score comes from treatment section
    s5 = sections.get("5_contradictions", {})
    s6 = sections.get("6_treatment", {})
    abx = r["result"].get("antibiotic_recommendation", {})

    # Check: no contraindicated drug prescribed
    allergies = r["case"].get("past_medical_history", {}).get("allergies", [])
    first_line = str(abx.get("first_line", "")).lower()
    safe = True
    for a in allergies:
        if isinstance(a, dict) and a.get("reaction_type") == "anaphylaxis":
            drug = a.get("drug", "").lower()
            if drug in first_line:
                safe = False

    status = "SAFE" if safe else "UNSAFE"
    print(f"  {'✓' if safe else '✗'} {case_id}: {status} — {abx.get('first_line', '?')}")

print(f"\n  → All cases should show SAFE — the system never prescribes contraindicated drugs")


In [ ]:
print("=== VERIFICATION CHECKLIST ===\n")

# Get results by case_id
def get_result(case_id):
    matches = [r for r in all_results if r["case_id"] == case_id]
    return matches[0] if matches else None

sepsis = get_result("SEPSIS-OVERRIDE")
missing = get_result("MISSING-UREA")
oral = get_result("ORAL-INTOLERANCE")
cr10_fire = get_result("CR10-FIRE")
cr10_true = get_result("CR10-TRUE")

checks = [
    ("SEPSIS-OVERRIDE: CURB65=1 (low severity)",
     sepsis and sepsis["result"].get("curb65_score", {}).get("curb65") == 1),
    ("SEPSIS-OVERRIDE: Sepsis override activated",
     sepsis and sepsis["result"].get("structured_output", {}).get("sections", {}).get("2_severity", {}).get("place_of_care", {}).get("sepsis_override", False)),
    ("CR10-FIRE: CR-10 contradiction fired",
     cr10_fire and any(c.get("rule_id") == "CR-10" for c in cr10_fire["result"].get("contradictions_detected", []))),
    ("CR10-TRUE: CR-10 did NOT fire (true allergy)",
     cr10_true and not any(c.get("rule_id") == "CR-10" for c in cr10_true["result"].get("contradictions_detected", []))),
    ("MISSING-UREA: CURB65 is None (urea missing)",
     missing and missing["result"].get("curb65_score", {}).get("curb65") is None),
    ("MISSING-UREA: CRB65 fallback used",
     missing and missing["result"].get("curb65_score", {}).get("crb65") is not None),
    ("ORAL-INTOLERANCE: IV route in antibiotic",
     oral and "IV" in str(oral["result"].get("antibiotic_recommendation", {}).get("first_line", ""))),
    ("All cases completed successfully",
     all(r["result"] is not None for r in all_results)),
]

passed = 0
for desc, ok in checks:
    status = "PASS" if ok else "FAIL"
    icon = "✓" if ok else "✗"
    print(f"  {icon} {status}: {desc}")
    if ok:
        passed += 1

print(f"\n{passed}/{len(checks)} checks passed")
